In [0]:
%sh
nc -zv c93366692d52454d9e959ac3f2dc9cb6.eastus.azure.elastic-cloud.com 443

c93366692d52454d9e959ac3f2dc9cb6.eastus.azure.elastic-cloud.com: forward host lookup failed: Host name lookup failure : Resource temporarily unavailable


In [0]:
%sh curl -s ifconfig.me

20.88.160.242

In [0]:
from sdds.common.util import NotebookUtil
from pyspark.sql.functions import col

from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType,
    ArrayType, IntegerType, LongType
)
from datetime import datetime

# Parameter widget for extraction date partition
dbutils.widgets.text("extraction_date", datetime.now().strftime("%Y-%m-%d"), "Extraction Date")
extraction_date = dbutils.widgets.get("extraction_date")

catalog_name=NotebookUtil.notebook_param("sdds_catalog")
bronze_schema_name=NotebookUtil.notebook_param("sdds_bronze_schema")
silver_schema_name=NotebookUtil.notebook_param("sdds_silver_schema")
bronze_table_name = "catalog-stream-dbx-bronze"
silver_table_name = "catalog-stream-dbx-silver"

In [0]:
from pyspark.sql.functions import col, from_json, explode_outer, lit, to_date, row_number, array_except, array, expr, try_element_at
from pyspark.sql.window import Window
from pyspark.sql.types import (
    ArrayType, StringType, StructType, StructField,
    DoubleType, LongType, IntegerType, MapType, FloatType  # FloatType was missing from the original imports
)

# --- Schema definitions for nested JSON fields ---
attributes_schema = ArrayType(MapType(StringType(), StringType()))

defAttributes_schema = ArrayType(StructType([
    StructField("identifier", StringType(), True),
    StructField("value", StringType(), True),
    StructField("seq", DoubleType(), True)
]))

customSkuAttributes_schema = ArrayType(StructType([
    StructField("key", StringType(), True),
    StructField("seq", FloatType(), True),
    StructField("storeId", LongType(), True),
    StructField("value", StringType(), True)
]))

stringFacets_schema = ArrayType(StructType([
    StructField("identifier", StringType(), True),
    StructField("storeId", LongType(), True),
    StructField("partnumber", StringType(), True),
    StructField("value", StringType(), True),
    StructField("seq", FloatType(), True),

]))

floatFacets_schema = ArrayType(StructType([
    StructField("identifier", StringType(), True),
    StructField("partnumber", StringType(), True),
    StructField("storeId", LongType(), True),
    StructField("seq", FloatType(), True),
    StructField("maxQty", FloatType(), True),
    StructField("minQty", FloatType(), True),
    StructField("value", FloatType(), True)
]))

numberFacets_schema = ArrayType(StructType([
    StructField("identifier", StringType(), True),
    StructField("partnumber", StringType(), True),
    StructField("storeId", LongType(), True),
    StructField("value", IntegerType(), True)
]))

priceList_schema = ArrayType(StructType([
    StructField("identifier", StringType(), True),
    StructField("stringValue", StringType(), True),
    StructField("startDateTime", LongType(), True),
    StructField("startDate", StringType(), True),
    StructField("endDateTime", LongType(), True),
    StructField("endDate", StringType(), True),
    StructField("value", DoubleType(), True),
    StructField("minQty", DoubleType(), True),
    StructField("maxQty", DoubleType(), True),
    StructField("partnumber", StringType(), True)
]))

seoURLs_schema = ArrayType(StructType([
    StructField("status", StringType(), True),
    StructField("url", StringType(), True),
    StructField("storeId", LongType(), True)
]))

seo_schema = StructType([
    StructField("asset", StructType([
        StructField("title", StringType(), True)
    ]), True),
    StructField("dsg", StructType([
        StructField("imageAltDesc", StringType(), True),
        StructField("metaDesc", StringType(), True),
        StructField("metaKeyword", StringType(), True),
        StructField("title", StringType(), True),
    ]), True),
    StructField("gg", StructType([
        StructField("imageAltDesc", StringType(), True),
        StructField("metaDesc", StringType(), True),
        StructField("metaKeyword", StringType(), True),
        StructField("title", StringType(), True),
    ]), True),
    StructField("pl", StructType([
        StructField("imageAltDesc", StringType(), True),
        StructField("metaDesc", StringType(), True),
        StructField("metaKeyword", StringType(), True),
        StructField("title", StringType(), True),
    ]), True),
])

productGroup_schema = ArrayType(StructType([
    StructField("id", StringType(), True),
    StructField("seq", LongType(), True),
    StructField("sequence", LongType(), True)
]))

primaryCategories_schema = StructType([
    StructField("dsg", StructType([
        StructField("id", StringType(), True),
        StructField("identifier", StringType(), True)
    ]), True),
    StructField("g3", StructType([
        StructField("id", StringType(), True),
        StructField("identifier", StringType(), True)
    ]), True),
    StructField("gg", StructType([
        StructField("id", StringType(), True),
        StructField("identifier", StringType(), True)
    ]), True),
    StructField("pl", StructType([
        StructField("id", StringType(), True),
        StructField("identifier", StringType(), True)
    ]), True),
])

salesData_schema = ArrayType(StructType([
    StructField("channel", StringType(), True),
    StructField("storeId", LongType(), True),
    StructField("costDollars", DoubleType(), True),
    StructField("margin", DoubleType(), True),
    StructField("salesDollars", DoubleType(), True),
    StructField("qtySold", LongType(), True)
]))

kafkaPriceList_schema = ArrayType(StructType([
    StructField("identifier", StringType(), True),
    StructField("partnumber", StringType(), True),
    StructField("stringValue", StringType(), True),
    StructField("startDateTime", LongType(), True),
    StructField("endDateTime", LongType(), True),
    StructField("maxQty", FloatType(), True),
    StructField("minQty", FloatType(), True),
    StructField("value", FloatType(), True)
]))

# Generic "price indicators" shape — reused for dsgPriceIndicators, ggPriceIndicators AND plPriceIndicators
dsgPriceIndicators_schema = StructType([
    StructField("dealsPercentage", DoubleType(), True),
    StructField("mapPriceIndicator", IntegerType(), True),
    StructField("priceIndicator", IntegerType(), True)
])

leafCategories_schema = ArrayType(StringType())

catgroupSeq_schema = ArrayType(StructType([
    StructField("key", StringType(), True),
    StructField("seq", DoubleType(), True)
]))

color_schema = StructType([
    StructField("swatch", StringType(), True),
    StructField("family", StringType(), True),
    StructField("seq", DoubleType(), True)
])

# Generic "overrides" shape — reused for dsgOverrides, ggOverrides AND plOverrides
overrides_schema = StructType([
    StructField("auxdescription1", StringType(), True),
    StructField("auxdescription2", StringType(), True),
    StructField("fullimage", StringType(), True),
    StructField("longdescription", StringType(), True),
    StructField("name", StringType(), True),
    StructField("published", LongType(), True),
    StructField("thumbnail", StringType(), True),
])

# dsg.brand / gg.cost / pl.savings etc. actually live under a top-level "ranking" object
ranking_brand_schema = StructType([      # reuse for ranking.dsg, ranking.gg, ranking.pl
    StructField("brand", FloatType(), True),
    StructField("cheap", FloatType(), True),
    StructField("clearance", FloatType(), True),
    StructField("cost", FloatType(), True),
    StructField("expensive", FloatType(), True),
    StructField("newness", FloatType(), True),
    StructField("ratings", FloatType(), True),
    StructField("sale", FloatType(), True),
    StructField("salesDollars", FloatType(), True),
    StructField("salesMargin", FloatType(), True),
    StructField("salesProfit", FloatType(), True),
    StructField("salesQuantity", FloatType(), True),
    StructField("salesTotalDollars", FloatType(), True),
    StructField("salesTotalMargin", FloatType(), True),
    StructField("salesTotalProfit", FloatType(), True),
    StructField("salesTotalQuantity", FloatType(), True),
    StructField("savings", FloatType(), True),
    StructField("verticalBrand", FloatType(), True),
])
RANKING_FIELDS = [f.name for f in ranking_brand_schema.fields]

# NOTE: ranking_schema was previously defined twice in the original script (once as a
# MapType, once as this StructType). Only this one is kept, since it's the shape that
# matches the fixed dsg/gg/pl fields you actually asked to structure. Because it's now a
# fixed-shape struct rather than a dynamic map, it's flattened directly in silver_df below
# instead of double-exploded into a separate silver_ranking_df.
ranking_schema = StructType([
    StructField("dsg", ranking_brand_schema, True),
    StructField("gg", ranking_brand_schema, True),
    StructField("pl", ranking_brand_schema, True),
])

# ============================================================================
# KAFKA PRICE LIST PIVOT CONFIGURATION
# ============================================================================

# Define the identifiers we expect in kafkaPriceList
KAFKA_PRICE_IDENTIFIERS = [
    "dickssportinggoodsmapprice",
    "dickssportinggoodslistprice",
    "dickssportinggoodsofferprice",
    "golfgalaxymapprice",
    "golfgalaxylistprice",
    "golfgalaxyofferprice",
    "publiclandsmapprice",
    "publiclandslistprice",
    "publiclandsofferprice"
]

# Define which fields from each price entry should become columns
KAFKA_PRICE_FIELDS = [
    "startDateTime",
    "endDateTime",
    "value",
    "stringValue",
    "minQty",
    "maxQty"
]


def add_kafka_price_pivot_columns(df):
    """
    Pivot kafkaPriceList array into flat columns.

    Creates columns with pattern: kafkaPriceList_identifier_<identifier>_<field>
    Example: kafkaPriceList_identifier_dickssportinggoodsmapprice_startDateTime

    For each identifier in KAFKA_PRICE_IDENTIFIERS and each field in KAFKA_PRICE_FIELDS,
    this filters the kafkaPriceList array for matching identifier and extracts the field value.
    Returns NULL if no match is found.
    """
    for identifier in KAFKA_PRICE_IDENTIFIERS:
        for field in KAFKA_PRICE_FIELDS:
            col_name = f"kafkaPriceList_identifier_{identifier}_{field}"

            # Expression to filter array for matching identifier and extract the field
            # Uses element_at with index 1 (1-based) to get first matching element
            # Returns null if no match found
            expr_str = f"""
                CASE
                    WHEN size(filter(kafkaPriceList, x -> x.identifier = '{identifier}')) > 0
                    THEN element_at(filter(kafkaPriceList, x -> x.identifier = '{identifier}'), 1).{field}
                    ELSE NULL
                END
            """

            df = df.withColumn(col_name, expr(expr_str))

    return df


# ============================================================================
# END KAFKA PRICE LIST PIVOT CONFIGURATION
# ============================================================================

# ============================================================================
# SALES DATA PIVOT CONFIGURATION
# ============================================================================

# Define the channels we expect in salesData
SALES_DATA_CHANNELS = [
    "ecomm",
    "total"
]

# Define the storeIds we expect in salesData
SALES_DATA_STORE_IDS = [
    10701,   # DSG
    11201,   # GG
    15108,   # PL
    16066    # Add other store IDs as needed
]

# Define which fields from each sales entry should become columns
SALES_DATA_FIELDS = [
    "costDollars",
    "margin",
    "salesDollars",
    "qtySold"
]


def add_sales_data_pivot_columns(df):
    """
    Pivot salesData array into flat columns.

    Creates columns with pattern: salesData_<channel>_<storeId>_<field>
    Example: salesData_ecomm_10701_costDollars

    For each combination of channel and storeId, this filters the salesData array
    for matching channel AND storeId and extracts the field value.
    Returns NULL if no match is found.
    """
    for channel in SALES_DATA_CHANNELS:
        for store_id in SALES_DATA_STORE_IDS:
            for field in SALES_DATA_FIELDS:
                col_name = f"salesData_{channel}_{store_id}_{field}"

                # Expression to filter array for matching channel AND storeId, then extract the field
                # Uses element_at with index 1 (1-based) to get first matching element
                # Returns null if no match found
                expr_str = f"""
                    CASE
                        WHEN size(filter(salesData, x -> x.channel = '{channel}' AND x.storeId = {store_id})) > 0
                        THEN element_at(filter(salesData, x -> x.channel = '{channel}' AND x.storeId = {store_id}), 1).{field}
                        ELSE NULL
                    END
                """

                df = df.withColumn(col_name, expr(expr_str))

    return df


# ============================================================================
# END SALES DATA PIVOT CONFIGURATION
# ============================================================================

# --- Read bronze (filtered by extraction_date partition) and deduplicate by partnumber ---
bronze_raw_df = (
    spark.read.table(f"`{catalog_name}`.`{bronze_schema_name}`.`{bronze_table_name}`")
    .filter(col("extraction_date") == extraction_date)
)
_w = Window.partitionBy("partnumber").orderBy(col("load_timestamp").desc())
bronze_df = (
    bronze_raw_df
    .withColumn("_rn", row_number().over(_w))
    .filter(col("_rn") == 1)
    .drop("_rn")
)
print(f"Reading bronze partition for extraction_date = {extraction_date}")
print(f"Bronze records (raw): {bronze_raw_df.count():,}  |  after dedup: {bronze_df.count():,}")

# --- Step 1: Parse all nested JSON into typed columns ---
parsed_df = (
    bronze_df
    .select(
        col("partnumber"),
        col("parentPartnumber"),
        col("type"),
        col("buyable").cast("int").cast("boolean").alias("buyable"),
        col("name"),
        col("mfName"),
        col("keyword"),
        col("longDescription"),
        col("thumbnail"),
        col("fullImage"),
        col("published").cast("int").cast("boolean").alias("published"),
        col("assetSeoUrl"),
        col("dsgSeoUrl"),
        col("plSeoUrl"),                       # ADDED
        col("taxCode"),
        col("productType"),
        col("startDate"),
        col("endDate"),
        from_json(col("kafkaPriceList"), kafkaPriceList_schema).alias("kafkaPriceList"),
        col("startDateTime"),
        col("endDateTime"),
        col("webActiveDate"),
        col("dsgProductSortDate"),
        col("plProductSortDate"),              # ADDED
        col("productSearchFlag").cast("int").cast("boolean").alias("productSearchFlag"),
        from_json(col("catalogIds"), ArrayType(StringType())).alias("catalogIds"),
        col("catentryId"),
        col("parentCatentryId"),
        col("primaryUPC"),
        col("searchAttributes"),
        col("auxDescription2"),
        col("dsgPublishOverride"),
        col("plPublishOverride"),              # ADDED
        col("onOrder"),
        col("comingSoonEndDateTime"),
        from_json(col("attributes"), attributes_schema).alias("attributes"),
        from_json(col("defAttributes"), defAttributes_schema).alias("defAttributes"),
        from_json(col("customSkuAttributes"), customSkuAttributes_schema).alias("customSkuAttributes"),
        from_json(col("stringFacets"), stringFacets_schema).alias("stringFacets"),
        from_json(col("floatFacets"), floatFacets_schema).alias("floatFacets"),
        from_json(col("numberFacets"), numberFacets_schema).alias("numberFacets"),
        from_json(col("priceList"), priceList_schema).alias("priceList"),
        from_json(col("seoURLs"), seoURLs_schema).alias("seoURLs"),
        from_json(col("seo"), seo_schema).alias("_seo"),                              # ADDED
        from_json(col("productGroup"), productGroup_schema).alias("productGroup"),
        from_json(col("primaryCategories"), primaryCategories_schema).alias("primaryCategories"),
        from_json(col("salesData"), salesData_schema).alias("salesData"),
        from_json(col("dsgPriceIndicators"), dsgPriceIndicators_schema).alias("_dsgPriceIndicators"),
        from_json(col("leafCategories"), leafCategories_schema).alias("leafCategories"),
        from_json(col("dsgCatgroups"), leafCategories_schema).alias("dsgCatgroups"),
        from_json(col("ggCatgroups"), leafCategories_schema).alias("ggCatgroups"),
        from_json(col("plCatgroups"), leafCategories_schema).alias("plCatgroups"),
        col("parentCatgroup"),
        col("parentCatgroup0"),
        col("parentCatgroup1"),
        col("parentCatgroup2"),
        col("parentCatgroup3"),
        col("parentCatgroup4"),
        col("parentCatgroup5"),
        col("parentCatgroup6"),
        col("parentCatgroup7"),
        col("parentCatgroup8"),
        col("parentCatgroup9"),
        from_json(col("catgroupSeq"), catgroupSeq_schema).alias("catgroupSeq"),
        from_json(col("ranking"), ranking_schema).alias("_ranking"),                  # was aliased "ranking"; renamed for flatten step below
        col("swatchPartNumber"),
        from_json(col("color"), color_schema).alias("_color"),
        # dsg/gg/pl overrides now all use the full overrides_schema (fullimage/thumbnail were
        # previously unreachable because dsgOverrides/ggOverrides used the narrower schema)
        from_json(col("dsgOverrides"), overrides_schema).alias("_dsgOverrides"),      # FIXED (was dsgOverrides_schema)
        from_json(col("ggOverrides"), overrides_schema).alias("_ggOverrides"),        # FIXED (was dsgOverrides_schema)
        from_json(col("plOverrides"), overrides_schema).alias("_plOverrides"),        # ADDED
        # GG nested structs (same shape as dsg equivalents)
        from_json(col("ggPriceIndicators"), dsgPriceIndicators_schema).alias("_ggPriceIndicators"),
        from_json(col("plPriceIndicators"), dsgPriceIndicators_schema).alias("_plPriceIndicators"),  # ADDED
        col("dsgQuantitySold").cast("long").alias("dsgQuantitySold"),
        col("ggQuantitySold").cast("long").alias("ggQuantitySold"),
        col("plQuantitySold").cast("long").alias("plQuantitySold"),
        col("dsgTotalPriceSold").cast("double").alias("dsgTotalPriceSold"),
        col("ggTotalPriceSold").cast("double").alias("ggTotalPriceSold"),
        col("plTotalPriceSold").cast("double").alias("plTotalPriceSold"),
        # GG scalar fields
        col("ggAkamaiRedirect").cast("boolean").alias("ggAkamaiRedirect"),
        col("ggAppWebActive").cast("int").cast("boolean").alias("ggAppWebActive"),
        col("ggKeywordOverride"),
        col("ggMobileAppWebActive").cast("int").cast("boolean").alias("ggMobileAppWebActive"),
        col("ggProductSortDate"),
        col("ggPublishOverride"),
        col("ggSeoUrl"),
        col("ggUrl"),
        col("ggWebActive").cast("int").cast("boolean").alias("ggWebActive"),
        # WebActive flags for the other banners — only the gg* versions existed before  (ADDED block)
        col("caliaWebActive").cast("int").cast("boolean").alias("caliaWebActive"),
        col("dsgAppWebActive").cast("int").cast("boolean").alias("dsgAppWebActive"),
        col("dsgMobileAppWebActive").cast("int").cast("boolean").alias("dsgMobileAppWebActive"),
        col("dsgWebActive").cast("int").cast("boolean").alias("dsgWebActive"),
        col("g3WebActive").cast("int").cast("boolean").alias("g3WebActive"),
        col("plWebActive").cast("int").cast("boolean").alias("plWebActive"),
        col("stackdWebActive").cast("int").cast("boolean").alias("stackdWebActive"),
        col("vrstWebActive").cast("int").cast("boolean").alias("vrstWebActive"),
        col("load_timestamp")
    )
)

# ============================================================================
# Apply kafkaPriceList pivoting transformation
# ============================================================================
print("Applying kafkaPriceList pivot transformation...")
parsed_df = add_kafka_price_pivot_columns(parsed_df)
print(f"Added {len(KAFKA_PRICE_IDENTIFIERS) * len(KAFKA_PRICE_FIELDS)} pivoted kafkaPriceList columns")

# ============================================================================
# Apply salesData pivoting transformation
# ============================================================================
print("Applying salesData pivot transformation...")
parsed_df = add_sales_data_pivot_columns(parsed_df)
print(f"Added {len(SALES_DATA_CHANNELS) * len(SALES_DATA_STORE_IDS) * len(SALES_DATA_FIELDS)} pivoted salesData columns")

# --- Step 2: Main silver table (scalar + flattened structs/maps, NO arrays) ---
# Add extraction_date as a DATE column (date-only, used for partitioning)
silver_df = (
    parsed_df
    .select(
        "partnumber", "parentPartnumber", "type", "buyable", "name", "mfName",
        "keyword", "longDescription", "thumbnail", "fullImage", "published",
        "assetSeoUrl", "dsgSeoUrl", "plSeoUrl", "taxCode", "productType",     # plSeoUrl ADDED
        "startDate", "endDate", "startDateTime", "endDateTime",
        "webActiveDate", "dsgProductSortDate", "plProductSortDate",          # plProductSortDate ADDED
        "productSearchFlag",
        "catalogIds", "catentryId", "parentCatentryId", "primaryUPC",
        "searchAttributes", "auxDescription2",
        "dsgPublishOverride", "plPublishOverride",                          # plPublishOverride ADDED
        "onOrder", "comingSoonEndDateTime",
        "attributes","customSkuAttributes","defAttributes",
        "floatFacets","stringFacets","numberFacets",
        # Flattened dsgPriceIndicators
        col("_dsgPriceIndicators.dealsPercentage").alias("dsgPriceIndicators_dealsPercentage"),
        col("_dsgPriceIndicators.mapPriceIndicator").alias("dsgPriceIndicators_mapPriceIndicator"),
        col("_dsgPriceIndicators.priceIndicator").alias("dsgPriceIndicators_priceIndicator"),
        # Flattened ggPriceIndicators
        col("_ggPriceIndicators.dealsPercentage").alias("ggPriceIndicators_dealsPercentage"),
        col("_ggPriceIndicators.mapPriceIndicator").alias("ggPriceIndicators_mapPriceIndicator"),
        col("_ggPriceIndicators.priceIndicator").alias("ggPriceIndicators_priceIndicator"),
        # Flattened plPriceIndicators (ADDED)
        col("_plOverrides.auxdescription1").alias("plOverrides_auxdescription1"),
        col("_plOverrides.auxdescription2").alias("plOverrides_auxdescription2"),
        col("_plOverrides.fullimage").alias("plOverrides_fullimage"),
        col("_plOverrides.name").alias("plOverrides_name"),
        col("_plOverrides.published").alias("plOverrides_published"),
        col("_plOverrides.thumbnail").alias("plOverrides_thumbnail"),
        col("_plOverrides.longdescription").alias("plOverrides_longdescription"),
        # Flattened primaryCategories (known keys per schema: dsg, g3, gg, pl)
        col("primaryCategories")["dsg"]["id"].alias("primaryCategories_dsg_id"),
        col("primaryCategories")["dsg"]["identifier"].alias("primaryCategories_dsg_identifier"),
        col("primaryCategories")["pl"]["id"].alias("primaryCategories_pl_id"),
        col("primaryCategories")["pl"]["identifier"].alias("primaryCategories_pl_identifier"),
        col("primaryCategories")["gg"]["id"].alias("primaryCategories_gg_id"),
        col("primaryCategories")["gg"]["identifier"].alias("primaryCategories_gg_identifier"),
        col("primaryCategories")["g3"]["id"].alias("primaryCategories_g3_id"),           # FIXED (was "fns", not in schema)
        col("primaryCategories")["g3"]["identifier"].alias("primaryCategories_g3_identifier"),  # FIXED
        
        # Flattened seo
        col("_seo.asset.title").alias("seo_asset_title"),
        col("_seo.dsg.imageAltDesc").alias("seo_dsg_imageAltDesc"),
        col("_seo.dsg.metaDesc").alias("seo_dsg_metaDesc"),
        col("_seo.dsg.metaKeyword").alias("seo_dsg_metaKeyword"),
        col("_seo.dsg.title").alias("seo_dsg_title"),
        col("_seo.gg.imageAltDesc").alias("seo_gg_imageAltDesc"),
        col("_seo.gg.metaDesc").alias("seo_gg_metaDesc"),
        col("_seo.gg.metaKeyword").alias("seo_gg_metaKeyword"),
        col("_seo.gg.title").alias("seo_gg_title"),
        col("_seo.pl.imageAltDesc").alias("seo_pl_imageAltDesc"),
        col("_seo.pl.metaDesc").alias("seo_pl_metaDesc"),
        col("_seo.pl.metaKeyword").alias("seo_pl_metaKeyword"),
        col("_seo.pl.title").alias("seo_pl_title"),                                                        # ADDED
        
        # Flattened productGroup (array — taking first element; see caveat above)
        try_element_at(col("productGroup"), lit(1))["id"].alias("productGroup_id"),
        try_element_at(col("productGroup"), lit(1))["seq"].alias("productGroup_seq"),
        try_element_at(col("productGroup"), lit(1))["sequence"].alias("productGroup_sequence"),
        
        # Flattened color
        col("_color.swatch").alias("color_swatch"),
        col("_color.family").alias("color_family"),
        col("_color.seq").alias("color_seq"),
        
        # Flattened dsgOverrides (fullimage/thumbnail now reachable — schema was fixed above)
        col("_dsgOverrides.auxdescription1").alias("dsgOverrides_auxdescription1"),                    # ADDED
        col("_dsgOverrides.auxdescription2").alias("dsgOverrides_auxdescription2"),                    # ADDED
        col("_dsgOverrides.fullimage").alias("dsgOverrides_fullimage"),                  # ADDED
        col("_dsgOverrides.name").alias("dsgOverrides_name"),
        col("_dsgOverrides.published").alias("dsgOverrides_published"),
        col("_dsgOverrides.thumbnail").alias("dsgOverrides_thumbnail"),                  # ADDED
        col("_dsgOverrides.longdescription").alias("dsgOverrides_longdescription"),                    # ADDED

        # Flattened ggOverrides (fullimage/thumbnail now reachable — schema was fixed above)
        col("_ggOverrides.auxdescription1").alias("ggOverrides_auxdescription1"),                    # ADDED
        col("_ggOverrides.auxdescription2").alias("ggOverrides_auxdescription2"),                    # ADDED
        col("_ggOverrides.fullimage").alias("ggOverrides_fullimage"),                    # ADDED
        col("_ggOverrides.name").alias("ggOverrides_name"),
        col("_ggOverrides.published").alias("ggOverrides_published"),
        col("_ggOverrides.thumbnail").alias("ggOverrides_thumbnail"),                    # ADDED
        col("_ggOverrides.longdescription").alias("ggOverrides_longdescription"),                    # ADDED
        # Other scalar fields
        "leafCategories", "dsgCatgroups", "ggCatgroups", "plCatgroups",
        # FIXED: was the single (nonexistent) column `parentCatgroup0-9`; expanded to the 10 real columns
        "catgroupSeq","parentCatgroup","parentCatgroup0", "parentCatgroup1", "parentCatgroup2", "parentCatgroup3",
        "parentCatgroup4", "parentCatgroup5", "parentCatgroup6", "parentCatgroup7",
        "parentCatgroup8", "parentCatgroup9",
        "swatchPartNumber","productGroup",
        "dsgQuantitySold", "ggQuantitySold", "plQuantitySold",
        "dsgTotalPriceSold", "ggTotalPriceSold", "plTotalPriceSold",
        # GG scalar fields
        "ggAkamaiRedirect", "ggAppWebActive", "ggKeywordOverride",
        "ggMobileAppWebActive", "ggProductSortDate", "ggPublishOverride",
        "ggSeoUrl", "ggUrl", "ggWebActive",
        # WebActive flags for the other banners (ADDED)
        "caliaWebActive", "dsgAppWebActive", "dsgMobileAppWebActive", "dsgWebActive",
        "g3WebActive", "plWebActive", "stackdWebActive", "vrstWebActive",
        # Flattened ranking (ADDED — replaces the old map-based silver_ranking_df since
        # ranking is now a fixed dsg/gg/pl struct rather than a dynamic map)
        *[col(f"_ranking.dsg.{f}").alias(f"ranking_dsg_{f}") for f in RANKING_FIELDS],
        *[col(f"_ranking.gg.{f}").alias(f"ranking_gg_{f}") for f in RANKING_FIELDS],
        *[col(f"_ranking.pl.{f}").alias(f"ranking_pl_{f}") for f in RANKING_FIELDS],
        # ============================================================================
        # PIVOTED KAFKA PRICE LIST COLUMNS
        # ============================================================================
        *[f"kafkaPriceList_identifier_{identifier}_{field}"
          for identifier in KAFKA_PRICE_IDENTIFIERS
          for field in KAFKA_PRICE_FIELDS],
        # ============================================================================
        # PIVOTED SALES DATA COLUMNS
        # ============================================================================
        *[f"salesData_{channel}_{store_id}_{field}"
          for channel in SALES_DATA_CHANNELS
          for store_id in SALES_DATA_STORE_IDS
          for field in SALES_DATA_FIELDS],
        # ============================================================================
        "load_timestamp",
        to_date(col("load_timestamp")).alias("extraction_date")
    )
)

# --- Step 3: Exploded tables (one row per array element, flat columns) ---

# attributes: Array<Map<String,String>> → partnumber | attribute_key | attribute_value
silver_attributes_df = (
    parsed_df.select("partnumber", explode_outer(col("attributes")).alias("attr_map"))
    .select("partnumber", explode_outer(col("attr_map")).alias("attribute_key", "attribute_value"))
)

# defAttributes: Array<Struct> → partnumber | identifier | value | seq
silver_defAttributes_df = (
    parsed_df.select("partnumber", explode_outer(col("defAttributes")).alias("elem"))
    .select("partnumber", col("elem.identifier"), col("elem.value"), col("elem.seq"))
)

# customSkuAttributes: Array<Struct> → partnumber | storeId | value | key
silver_customSkuAttributes_df = (
    parsed_df.select("partnumber", explode_outer(col("customSkuAttributes")).alias("elem"))
    .select("partnumber", col("elem.storeId"), col("elem.value"), col("elem.key"))
)

# stringFacets: Array<Struct> → partnumber | identifier | storeId | facet_partnumber | value
silver_stringFacets_df = (
    parsed_df.select("partnumber", explode_outer(col("stringFacets")).alias("elem"))
    .select(
        "partnumber",
        col("elem.identifier"),
        col("elem.storeId"),
        col("elem.partnumber").alias("facet_partnumber"),
        col("elem.value")
    )
)

# floatFacets: Array<Struct> → partnumber | identifier | facet_partnumber | value | storeId | maxQty | minQty
silver_floatFacets_df = (
    parsed_df.select("partnumber", explode_outer(col("floatFacets")).alias("elem"))
    .select(
        "partnumber",
        col("elem.identifier"),
        col("elem.partnumber").alias("facet_partnumber"),
        col("elem.value"),
        col("elem.storeId"),
        col("elem.maxQty"),
        col("elem.minQty")
    )
)

# numberFacets: Array<Struct> → partnumber | identifier | facet_partnumber | value | storeId  (ADDED — was parsed but never exploded)
silver_numberFacets_df = (
    parsed_df.select("partnumber", explode_outer(col("numberFacets")).alias("elem"))
    .select(
        "partnumber",
        col("elem.identifier"),
        col("elem.partnumber").alias("facet_partnumber"),
        col("elem.value"),
        col("elem.storeId")
    )
)

# priceList: Array<Struct> → partnumber | identifier | stringValue | startDateTime | startDate | endDateTime | endDate | value | minQty | maxQty | price_partnumber
silver_priceList_df = (
    parsed_df.select("partnumber", explode_outer(col("priceList")).alias("elem"))
    .select(
        "partnumber",
        col("elem.identifier"),
        col("elem.stringValue"),
        col("elem.startDateTime"),
        col("elem.startDate"),
        col("elem.endDateTime"),
        col("elem.endDate"),
        col("elem.value"),
        col("elem.minQty"),
        col("elem.maxQty"),
        col("elem.partnumber").alias("price_partnumber")
    )
)

# kafkaPriceList: Array<Struct> → partnumber | identifier | stringValue | startDateTime | endDateTime | value | minQty | maxQty | kafka_partnumber
# NOTE: Keeping this exploded table for flexibility (ad-hoc queries on any identifier)
# The pivoted columns in silver_df provide fast access to key identifiers
silver_kafkaPriceList_df = (
    parsed_df.select("partnumber", explode_outer(col("kafkaPriceList")).alias("elem"))
    .select(
        "partnumber",
        col("elem.identifier"),
        col("elem.stringValue"),
        col("elem.startDateTime"),
        col("elem.endDateTime"),
        col("elem.value"),
        col("elem.minQty"),
        col("elem.maxQty"),
        col("elem.partnumber").alias("kafka_partnumber")
    )
)

# seoURLs: Array<Struct> → partnumber | status | url | storeId
silver_seoURLs_df = (
    parsed_df.select("partnumber", explode_outer(col("seoURLs")).alias("elem"))
    .select("partnumber", col("elem.status"), col("elem.url"), col("elem.storeId"))
)

# productGroup: Array<Struct> → partnumber | group_id | sequence
silver_productGroup_df = (
    parsed_df.select("partnumber", explode_outer(col("productGroup")).alias("elem"))
    .select(
        "partnumber",
        col("elem.id").alias("group_id"),
        col("elem.seq"),
        col("elem.sequence")
    )
)

# salesData: Array<Struct> → partnumber | channel | storeId | costDollars | margin | salesDollars | qtySold
silver_salesData_df = (
    parsed_df.select("partnumber", explode_outer(col("salesData")).alias("elem"))
    .select(
        "partnumber",
        col("elem.channel"),
        col("elem.storeId"),
        col("elem.costDollars"),
        col("elem.margin"),
        col("elem.salesDollars"),
        col("elem.qtySold")
    )
)

# catgroupSeq: Array<Struct> → partnumber | catgroup_key | seq
silver_catgroupSeq_df = (
    parsed_df.select("partnumber", explode_outer(col("catgroupSeq")).alias("elem"))
    .select("partnumber", col("elem.key").alias("catgroup_key"), col("elem.seq"))
)

# NOTE: silver_ranking_df has been removed. ranking is now a fixed dsg/gg/pl struct
# (see ranking_schema above), not a dynamic map, so it's flattened directly into
# silver_df as ranking_dsg_*, ranking_gg_*, ranking_pl_* columns instead of being
# exploded into its own table.

print("=== Silver DataFrames created ===")
print(f"  silver_df (main):             {silver_df.columns.__len__()} columns")
print(f"  silver_attributes_df:         {silver_attributes_df.columns}")
print(f"  silver_defAttributes_df:      {silver_defAttributes_df.columns}")
print(f"  silver_customSkuAttributes_df:{silver_customSkuAttributes_df.columns}")
print(f"  silver_stringFacets_df:       {silver_stringFacets_df.columns}")
print(f"  silver_floatFacets_df:        {silver_floatFacets_df.columns}")
print(f"  silver_numberFacets_df:       {silver_numberFacets_df.columns}")
print(f"  silver_priceList_df:          {silver_priceList_df.columns}")
print(f"  silver_kafkaPriceList_df:     {silver_kafkaPriceList_df.columns}")
print(f"  silver_seoURLs_df:            {silver_seoURLs_df.columns}")
print(f"  silver_productGroup_df:       {silver_productGroup_df.columns}")
print(f"  silver_salesData_df:          {silver_salesData_df.columns}")
print(f"  silver_catgroupSeq_df:        {silver_catgroupSeq_df.columns}")

# ============================================================================
# Verify kafkaPriceList pivot columns
# ============================================================================
print("\n=== Verifying kafkaPriceList pivot columns ===")
kafka_price_cols = [c for c in silver_df.columns if c.startswith("kafkaPriceList_identifier_")]
print(f"Found {len(kafka_price_cols)} kafkaPriceList pivot columns")
print("Sample columns:")
for col in kafka_price_cols[:5]:
    print(f"  - {col}")

# ============================================================================
# Verify salesData pivot columns
# ============================================================================
print("\n=== Verifying salesData pivot columns ===")
sales_data_cols = [c for c in silver_df.columns if c.startswith("salesData_")]
print(f"Found {len(sales_data_cols)} salesData pivot columns")
print("Sample columns:")
for col in sales_data_cols[:5]:
    print(f"  - {col}")


Reading bronze partition for extraction_date = 2026-09-01
Bronze records (raw): 1,804,738  |  after dedup: 1,804,675
Applying kafkaPriceList pivot transformation...
Added 54 pivoted kafkaPriceList columns
Applying salesData pivot transformation...
Added 32 pivoted salesData columns
=== Silver DataFrames created ===
  silver_df (main):             277 columns
  silver_attributes_df:         ['partnumber', 'attribute_key', 'attribute_value']
  silver_defAttributes_df:      ['partnumber', 'identifier', 'value', 'seq']
  silver_customSkuAttributes_df:['partnumber', 'storeId', 'value', 'key']
  silver_stringFacets_df:       ['partnumber', 'identifier', 'storeId', 'facet_partnumber', 'value']
  silver_floatFacets_df:        ['partnumber', 'identifier', 'facet_partnumber', 'value', 'storeId', 'maxQty', 'minQty']
  silver_numberFacets_df:       ['partnumber', 'identifier', 'facet_partnumber', 'value', 'storeId']
  silver_priceList_df:          ['partnumber', 'identifier', 'stringValue', 'start

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{silver_schema_name}`")

# --- Helper: prefix exploded columns to avoid name clashes on join ---
def prefix_cols(df, prefix, key_col="partnumber"):
    new_names = [c if c == key_col else f"{prefix}_{c}" for c in df.columns]
    return df.toDF(*new_names)

# --- Join each exploded table with main silver_df to enrich with product context ---
# Each table gets ALL main columns + prefixed exploded array columns (no cross-product)

silver_tables = {
    "": silver_df,
    #"-attributes": silver_df.join(prefix_cols(silver_attributes_df, "attr"), "partnumber", "left"),
    #"-def-attributes": silver_df.join(prefix_cols(silver_defAttributes_df, "defAttr"), "partnumber", "left"),
    #"-custom-sku-attributes": silver_df.join(prefix_cols(silver_customSkuAttributes_df, "customSku"), "partnumber", "left"),
    #"-string-facets": silver_df.join(prefix_cols(silver_stringFacets_df, "strFacet"), "partnumber", "left"),
    #"-float-facets": silver_df.join(prefix_cols(silver_floatFacets_df, "fltFacet"), "partnumber", "left"),
    #"-price-list": silver_df.join(prefix_cols(silver_priceList_df, "price"), "partnumber", "left"),
    #"-seo-urls": silver_df.join(prefix_cols(silver_seoURLs_df, "seo"), "partnumber", "left"),
    #"-product-group": silver_df.join(prefix_cols(silver_productGroup_df, "prodGroup"), "partnumber", "left"),
    #"-sales-data": silver_df.join(prefix_cols(silver_salesData_df, "sales"), "partnumber", "left"),
    #"-catgroup-seq": silver_df.join(prefix_cols(silver_catgroupSeq_df, "catSeq"), "partnumber", "left"),
    #"-kafka-price-list": silver_df.join(prefix_cols(silver_kafkaPriceList_df, "kafkaPrice"), "partnumber", "left"),
    #"-ranking": silver_df.join(prefix_cols(silver_ranking_df, "rank"), "partnumber", "left"),
}

for suffix, df in silver_tables.items():
    table_full = f"`{catalog_name}`.`{silver_schema_name}`.`{silver_table_name}{suffix}`"
    table_exists = spark.catalog.tableExists(f"{catalog_name}.{silver_schema_name}.`{silver_table_name}{suffix}`")

    if not table_exists:
        # First run: create table with extraction_date partition
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .partitionBy("extraction_date")
            .saveAsTable(table_full)
        )
    else:
        # Subsequent runs: overwrite only this partition (idempotent re-runs)
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("replaceWhere", f"extraction_date = '{extraction_date}'")
            .option("mergeSchema", "true")
            .partitionBy("extraction_date")
            .saveAsTable(table_full)
        )
    print(f"  Written {df.count():,} rows to {silver_table_name}{suffix}")

print("\nAll silver tables written successfully.")

  Written 1,804,675 rows to catalog-stream-dbx-silver

All silver tables written successfully.
